# 18wB — Development-only probability calibration

The selected baseline, Gaussian-process and tree distributions are recalibrated using development OOF information only. The calibrator has two transparent parameters:

\[
q^{\mathrm{cal}}_j=q_{0.50}+s(q_j-q_{0.50}),\qquad
p^{\mathrm{cal}}=(1-\eta)p^{\mathrm{particles}}+\eta/11.
\]

The scale factor widens or contracts the predictive distribution around its median. Uniform smoothing prevents brittle zero probabilities. Parameters are selected separately for each family representative by date-balanced categorical log score, then multiclass Brier score, then distance from the identity transform. No holdout or June outcome is loaded.

**Revision v2.** Calibration-grid scoring is vectorised over the 99 particles and caches the eleven-contract book boundaries, avoiding repeated DataFrame construction.

**Revision v3.** Contract definitions and development outcomes are inherited from the 18wA development panel. The full 18s outcome file is not opened.

In [1]:
from __future__ import annotations
import hashlib, json, math, platform, sys
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
ROOT=Path.cwd().resolve()
if not (ROOT/'.git').exists(): raise RuntimeError(f'Run from repository root, not {ROOT}')
UTC=timezone.utc; STEP='18wB'; EPS=1e-12; NQ=99; NC=11
VA=ROOT/'data/processed/18vA_residual_model_design_and_features'; VC=ROOT/'data/processed/18vC_common_support_scoring_and_selection'; WA=ROOT/'data/processed/18wA_contract_probability_mapping'
QGRID=VA/'18vA_quantile_grid.csv'; VA_MAN=VA/'18vA_sha256_manifest.csv'; VA_SUM=VA/'18vA_summary.json'
OOF=VC/'18vC_all_candidate_oof_predictions.csv'; SELECT=VC/'18vC_selected_candidates.json'; VC_MAN=VC/'18vC_sha256_manifest.csv'; VC_SUM=VC/'18vC_summary.json'
UNCAL=WA/'18wA_development_uncalibrated_probability_panel.csv'; UNCAL_SUM=WA/'18wA_development_uncalibrated_score_summary.csv'; WA_MAN=WA/'18wA_sha256_manifest.csv'; WA_SUM=WA/'18wA_summary.json'
OUT=ROOT/'data/processed/18wB_development_probability_calibration'; REPORT=ROOT/'reports/18wB_development_probability_calibration'; OUT.mkdir(parents=True,exist_ok=True); REPORT.mkdir(parents=True,exist_ok=True)
for p in [QGRID,VA_MAN,VA_SUM,OOF,SELECT,VC_MAN,VC_SUM,UNCAL,UNCAL_SUM,WA_MAN,WA_SUM]:
    if not p.is_file(): raise FileNotFoundError(p)
SCALES=np.array([0.75,1.00,1.25,1.50,1.75,2.00,2.50,3.00],float); ETAS=np.array([0.000,0.005,0.010,0.020,0.050,0.100],float)

def sha(p):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for c in iter(lambda:f.read(1024*1024),b''): h.update(c)
    return h.hexdigest()
def verify(p):
    bad=[]
    for r in pd.read_csv(p).itertuples(index=False):
        q=ROOT/r.path
        if not q.is_file(): bad.append('MISSING '+r.path); continue
        if sha(q)!=r.sha256: bad.append('HASH '+r.path)
        if q.stat().st_size!=int(r.size_bytes): bad.append('SIZE '+r.path)
    if bad: raise AssertionError('\n'.join(bad))
def pbool(s,name):
    if pd.api.types.is_bool_dtype(s): return s.astype(bool)
    x=s.astype(str).str.strip().str.lower().map({'true':True,'false':False,'1':True,'0':False,'yes':True,'no':False})
    if x.isna().any(): raise ValueError(f'Cannot parse {name}')
    return x.astype(bool)
def order_book(g):
    z=g.copy(); z['_e']=z.event_type.map({'lower':0,'interior':1,'upper':2}); z['_l']=z.lower_bound_c.fillna(-1e9); z=z.sort_values(['_e','_l','upper_bound_c','market_id'],na_position='last').drop(columns=['_e','_l']).reset_index(drop=True); z['contract_order']=np.arange(len(z)); return z
def member(x,r):
    if r.event_type=='lower': return x<float(r.upper_bound_c)
    if r.event_type=='interior': return (x>=float(r.lower_bound_c))&(x<float(r.upper_bound_c))
    if r.event_type=='upper': return x>=float(r.lower_bound_c)
    raise ValueError(r.event_type)
def map_prob(residual_quantiles,forecast,book,scale,eta):
    q=np.asarray(residual_quantiles,float); center=q[49]; particles=float(forecast)+center+float(scale)*(q-center); b=order_book(book); M=np.column_stack([member(particles,r) for r in b.itertuples(index=False)])
    if not np.all(M.sum(1)==1): raise AssertionError('Particle mapping')
    counts=M.sum(0).astype(int); raw=counts/NQ; calibrated=(1-float(eta))*raw+float(eta)/NC
    if counts.sum()!=NQ or not np.isclose(calibrated.sum(),1,atol=1e-12): raise AssertionError('Incoherent calibration')
    return b,counts,raw,calibrated
def date_weights(df):
    n=df.event_date.nunique(); w=1/(n*df.groupby('event_date').event_date.transform('size'))
    if not np.isclose(w.sum(),1,atol=1e-12): raise AssertionError('Weights')
    return w

for p in [VA_MAN,VC_MAN,WA_MAN]: verify(p)
for name,p in [('18vA',VA_SUM),('18vC',VC_SUM),('18wA',WA_SUM)]:
    if json.loads(p.read_text()).get('verdict')!='PASS': raise AssertionError(f'{name} is not PASS')
selection=json.loads(SELECT.read_text()); selected=[selection['selected_baseline'],selection['selected_gaussian_process'],selection['selected_tree']]
qgrid=pd.read_csv(QGRID); oof=pd.read_csv(OOF,low_memory=False); uncal=pd.read_csv(UNCAL,dtype={'market_id':str},low_memory=False); uncal_summary=pd.read_csv(UNCAL_SUM)
contracts=uncal.drop_duplicates(['event_date','market_id']).copy()
for df in [contracts,oof,uncal]: df['event_date']=pd.to_datetime(df.event_date,errors='raise')
oof['common_candidate_selection_support']=pbool(oof.common_candidate_selection_support,'common support')
rq=qgrid.residual_quantile_column.tolist(); books={d:order_book(g) for d,g in contracts.groupby('event_date')}
dev=oof[oof.candidate_id.isin(selected)&oof.common_candidate_selection_support].copy()
if len(dev)!=408 or not dev.groupby('candidate_id').size().eq(136).all(): raise AssertionError('Development support')

griddef=pd.MultiIndex.from_product([SCALES,ETAS],names=['scale_factor','uniform_smoothing']).to_frame(index=False); griddef['scale_distance_from_identity']=np.abs(np.log(griddef.scale_factor)); griddef['calibration_complexity_rank']=griddef.sort_values(['scale_distance_from_identity','uniform_smoothing','scale_factor']).reset_index().index+1
# Precompute ordered book boundaries and outcome vectors once.
boundaries={}
yvectors={}
for d,b in books.items():
    ob=order_book(b)
    boundaries[d]=ob.iloc[:10].upper_bound_c.to_numpy(float)
    yvectors[d]=ob.Y_event_int.to_numpy(float)

def probability_matrix_for_scale(group,scale):
    q=group[rq].to_numpy(float); centers=q[:,49:50]; particles=group.forecast_daily_max_c.to_numpy(float)[:,None]+centers+float(scale)*(q-centers)
    raw=np.empty((len(group),NC),float)
    for i,(d,row) in enumerate(zip(group.event_date,particles)):
        category=np.searchsorted(boundaries[d],row,side='right')
        raw[i]=np.bincount(category,minlength=NC)/NQ
    return raw

gridrows=[]
for cid,g in dev.groupby('candidate_id',sort=True):
    g=g.sort_values(['event_date','decision_rule_order']).reset_index(drop=True)
    weights=g.date_balanced_selection_weight.to_numpy(float)
    Y=np.vstack([yvectors[d] for d in g.event_date])
    winner_index=Y.argmax(axis=1)
    for scale in SCALES:
        raw=probability_matrix_for_scale(g,scale)
        for eta in ETAS:
            P=(1-float(eta))*raw+float(eta)/NC
            win=P[np.arange(len(g)),winner_index]
            logs=-np.log(np.clip(win,EPS,1.0))
            briers=np.square(P-Y).sum(axis=1)
            gridrows.append({'candidate_id':cid,'scale_factor':float(scale),'uniform_smoothing':float(eta),'development_books':len(g),'development_dates':g.event_date.nunique(),'date_balanced_mean_categorical_log_score':float(np.average(logs,weights=weights)),'date_balanced_mean_multiclass_brier':float(np.average(briers,weights=weights)),'zero_winning_probability_books':int((win==0).sum()),'minimum_winning_probability':float(win.min()),'scale_distance_from_identity':abs(math.log(float(scale)))})
grid=pd.DataFrame(gridrows); grid['identity_transform']=np.isclose(grid.scale_factor,1)&np.isclose(grid.uniform_smoothing,0)
grid=grid.sort_values(['candidate_id','date_balanced_mean_categorical_log_score','date_balanced_mean_multiclass_brier','scale_distance_from_identity','uniform_smoothing','scale_factor'],kind='mergesort').reset_index(drop=True); grid['within_candidate_rank']=grid.groupby('candidate_id').cumcount()+1
params=grid[grid.within_candidate_rank.eq(1)].copy().sort_values('candidate_id').reset_index(drop=True)
expected={'pooled_empirical_residual':(1.25,0.0),'gp_matern32_rule':(1.5,0.0),'catboost_quantile_pooled':(2.0,0.1)}
for r in params.itertuples(index=False):
    if r.candidate_id not in expected or not np.isclose(r.scale_factor,expected[r.candidate_id][0]) or not np.isclose(r.uniform_smoothing,expected[r.candidate_id][1]): raise AssertionError(f'Unexpected calibration selection {r}')

calparts=[]; score_rows=[]
for cid,g in dev.groupby('candidate_id',sort=True):
    pr=params[params.candidate_id.eq(cid)].iloc[0]; scale=float(pr.scale_factor); eta=float(pr.uniform_smoothing)
    for r in g.itertuples(index=False):
        b,counts,raw,p=map_prob(np.array([getattr(r,c) for c in rq],float),r.forecast_daily_max_c,books[r.event_date],scale,eta); z=b.copy(); z['particle_count_calibrated_mapping']=counts; z['p_particle_scaled']=raw; z['p_model_calibrated']=p
        for c,v in {'candidate_id':cid,'model_family':r.model_family,'scope_type':r.scope_type,'scope_id':r.scope_id,'decision_rule':r.decision_rule,'decision_rule_order':r.decision_rule_order,'development_fold':r.development_fold,'forecast_daily_max_c':r.forecast_daily_max_c,'calibration_scale_factor':scale,'calibration_uniform_smoothing':eta,'calibration_variant':'DEVELOPMENT_SELECTED_SCALE_UNIFORM','date_balanced_selection_weight':r.date_balanced_selection_weight}.items(): z[c]=v
        calparts.append(z)
        y=b.Y_event_int.to_numpy(float); win=float(p[y==1][0]); score_rows.append({'candidate_id':cid,'event_date':r.event_date,'decision_rule':r.decision_rule,'decision_rule_order':r.decision_rule_order,'development_fold':r.development_fold,'calibration_scale_factor':scale,'calibration_uniform_smoothing':eta,'winning_probability':win,'categorical_log_score':-math.log(min(max(win,EPS),1.0)),'multiclass_brier':float(np.square(p-y).sum()),'zero_winning_probability':win==0,'date_balanced_selection_weight':r.date_balanced_selection_weight})
calprob=pd.concat(calparts,ignore_index=True); calscores=pd.DataFrame(score_rows)
if len(calprob)!=4488 or len(calscores)!=408: raise AssertionError('Calibrated outputs')
if not calprob.groupby(['candidate_id','event_date','decision_rule']).p_model_calibrated.sum().apply(lambda v:np.isclose(v,1,atol=1e-12)).all(): raise AssertionError('Calibrated book sums')

compare=[]
for cid in selected:
    u=uncal_summary[uncal_summary.candidate_id.eq(cid)].iloc[0]; g=calscores[calscores.candidate_id.eq(cid)]; w=g.date_balanced_selection_weight.to_numpy(float); p=params[params.candidate_id.eq(cid)].iloc[0]
    compare += [
        {'candidate_id':cid,'calibration_variant':'UNCALIBRATED','scale_factor':1.0,'uniform_smoothing':0.0,'development_books':int(u.development_books),'date_balanced_mean_categorical_log_score':float(u.date_balanced_mean_categorical_log_score),'date_balanced_mean_multiclass_brier':float(u.date_balanced_mean_multiclass_brier),'zero_winning_probability_books':int(u.zero_winning_probability_books)},
        {'candidate_id':cid,'calibration_variant':'CALIBRATED','scale_factor':float(p.scale_factor),'uniform_smoothing':float(p.uniform_smoothing),'development_books':len(g),'date_balanced_mean_categorical_log_score':float(np.average(g.categorical_log_score,weights=w)),'date_balanced_mean_multiclass_brier':float(np.average(g.multiclass_brier,weights=w)),'zero_winning_probability_books':int(g.zero_winning_probability.sum())},
    ]
compare=pd.DataFrame(compare); wide=compare.pivot(index='candidate_id',columns='calibration_variant',values=['date_balanced_mean_categorical_log_score','date_balanced_mean_multiclass_brier']); improvements=[]
for cid in selected:
    u=compare[(compare.candidate_id.eq(cid))&(compare.calibration_variant.eq('UNCALIBRATED'))].iloc[0]; c=compare[(compare.candidate_id.eq(cid))&(compare.calibration_variant.eq('CALIBRATED'))].iloc[0]
    improvements.append({'candidate_id':cid,'categorical_log_improvement':float(u.date_balanced_mean_categorical_log_score-c.date_balanced_mean_categorical_log_score),'multiclass_brier_improvement':float(u.date_balanced_mean_multiclass_brier-c.date_balanced_mean_multiclass_brier),'calibration_improves_development_log':bool(c.date_balanced_mean_categorical_log_score<u.date_balanced_mean_categorical_log_score)})
improvements=pd.DataFrame(improvements)
if not improvements.calibration_improves_development_log.all(): raise AssertionError('Selected calibration does not improve log score')
checks=pd.DataFrame([
{'check':'calibration_grid_rows_144','passed':len(grid)==144,'detail':f'rows={len(grid)}','blocking':True},
{'check':'selected_parameter_rows_3','passed':len(params)==3,'detail':params[['candidate_id','scale_factor','uniform_smoothing']].to_dict('records').__str__(),'blocking':True},
{'check':'calibrated_probability_rows_4488','passed':len(calprob)==4488,'detail':f'rows={len(calprob)}','blocking':True},
{'check':'calibrated_book_scores_408','passed':len(calscores)==408,'detail':f'rows={len(calscores)}','blocking':True},
{'check':'all_selected_calibrations_improve_log','passed':improvements.calibration_improves_development_log.all(),'detail':improvements.to_dict('records').__str__(),'blocking':True},
{'check':'full_outcome_panel_not_loaded','passed':True,'detail':'contract definitions and development outcomes inherited from 18wA','blocking':True},
{'check':'holdout_external_outcomes_not_loaded','passed':dev.event_date.max()<=pd.Timestamp('2026-05-21'),'detail':'development only','blocking':True},
{'check':'market_information_absent','passed':'p_market' not in dev.columns,'detail':'weather model calibration only','blocking':True},
])
if not checks.passed.all(): raise AssertionError(checks[~checks.passed].to_string(index=False))
issues=pd.DataFrame(columns=['issue_level','issue_code','candidate_id','scale_factor','uniform_smoothing','detail','blocking'])
outputs={'18wB_calibration_grid_definition.csv':griddef,'18wB_calibration_grid_scores.csv':grid,'18wB_selected_calibration_parameters.csv':params,'18wB_development_calibrated_probability_panel.csv':calprob,'18wB_development_calibrated_book_scores.csv':calscores,'18wB_development_calibration_comparison.csv':compare,'18wB_calibration_improvements.csv':improvements,'18wB_integrity_checks.csv':checks,'18wB_issues.csv':issues}
for name,df in outputs.items():
    z=df.copy()
    for c in z.columns:
        if 'date' in c.lower() and pd.api.types.is_datetime64_any_dtype(z[c]): z[c]=z[c].dt.strftime('%Y-%m-%d')
        if 'cutoff' in c.lower() or c.lower().endswith('_utc') or 'available' in c.lower(): z[c]=z[c].astype('string')
    z.to_csv(OUT/name,index=False)
protocol={'step':STEP,'generated_at_utc':datetime.now(UTC).isoformat(),'verdict':'PASS','calibration_transform':'median-centred residual quantile scale plus uniform simplex smoothing','scale_grid':SCALES.tolist(),'uniform_smoothing_grid':ETAS.tolist(),'selection_hierarchy':['date_balanced_mean_categorical_log_score','date_balanced_mean_multiclass_brier','scale_distance_from_identity','uniform_smoothing','scale_factor'],'selected_parameters':params[['candidate_id','scale_factor','uniform_smoothing']].to_dict('records'),'development_common_books_per_candidate':136,'contract_definitions_and_outcomes_source':'18wA development probability panel','full_outcome_panel_loaded':False,'holdout_or_external_outcomes_loaded':False,'market_information_used':False,'probability_bridge_retained':False}
(OUT/'18wB_protocol.json').write_text(json.dumps(protocol,indent=2),encoding='utf-8')
sources=pd.DataFrame([
{'input_role':'18vA_quantile_grid','path':str(QGRID.relative_to(ROOT)),'rows':len(qgrid),'sha256':sha(QGRID)},
{'input_role':'18vC_all_candidate_oof','path':str(OOF.relative_to(ROOT)),'rows':len(oof),'sha256':sha(OOF)},
{'input_role':'18vC_selection','path':str(SELECT.relative_to(ROOT)),'rows':1,'sha256':sha(SELECT)},
{'input_role':'18wA_uncalibrated_probability_panel','path':str(UNCAL.relative_to(ROOT)),'rows':len(uncal),'sha256':sha(UNCAL)},
{'input_role':'18wA_uncalibrated_score_summary','path':str(UNCAL_SUM.relative_to(ROOT)),'rows':len(uncal_summary),'sha256':sha(UNCAL_SUM)},
]); sources.to_csv(OUT/'18wB_source_inventory.csv',index=False)
summary={'step':STEP,'generated_at_utc':datetime.now(UTC).isoformat(),'verdict':'PASS','selected_candidates':3,'calibration_grid_combinations_per_candidate':48,'calibration_grid_score_rows':144,'selected_parameter_rows':3,'development_books_per_candidate':136,'calibrated_contract_probability_rows':4488,'calibrated_book_score_rows':408,'full_outcome_panel_loaded':False,'holdout_or_external_outcomes_loaded':False,'market_information_used':False,'probability_bridge_retained':False,'selected_parameters':params[['candidate_id','scale_factor','uniform_smoothing']].to_dict('records'),'issue_rows':0,'integrity_checks_passed':int(checks.passed.sum()),'integrity_checks_total':len(checks)}
(OUT/'18wB_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
(OUT/'18wB_environment.json').write_text(json.dumps({'generated_at_utc':datetime.now(UTC).isoformat(),'python':sys.version,'platform':platform.platform(),'pandas':pd.__version__,'numpy':np.__version__,'revision':'v3'},indent=2),encoding='utf-8')
lines=['# 18wB development-only probability calibration','','**PASS**','','## Selected parameters','','| Candidate | Scale | Uniform smoothing | Development log | Development Brier |','|---|---:|---:|---:|---:|']
for r in params.itertuples(index=False): lines.append(f'| {r.candidate_id} | {r.scale_factor:.2f} | {r.uniform_smoothing:.3f} | {r.date_balanced_mean_categorical_log_score:.6f} | {r.date_balanced_mean_multiclass_brier:.6f} |')
lines += ['','All parameters were fitted and selected using the common development OOF support only. No holdout, June or market outcome was loaded.']
(REPORT/'18wB_development_probability_calibration_report.md').write_text('\n'.join(lines)+'\n',encoding='utf-8')
manifest=[]
for root in [OUT,REPORT]:
    for p in sorted(root.rglob('*')):
        if p.is_file() and p.name!='18wB_sha256_manifest.csv': manifest.append({'path':str(p.relative_to(ROOT)),'size_bytes':p.stat().st_size,'sha256':sha(p)})
pd.DataFrame(manifest).to_csv(OUT/'18wB_sha256_manifest.csv',index=False)
print(json.dumps(summary,indent=2)); display(params[['candidate_id','scale_factor','uniform_smoothing','date_balanced_mean_categorical_log_score','date_balanced_mean_multiclass_brier']]); print('18wB development calibration release: PASS')

{
  "step": "18wB",
  "generated_at_utc": "2026-07-22T12:27:01.775765+00:00",
  "verdict": "PASS",
  "selected_candidates": 3,
  "calibration_grid_combinations_per_candidate": 48,
  "calibration_grid_score_rows": 144,
  "selected_parameter_rows": 3,
  "development_books_per_candidate": 136,
  "calibrated_contract_probability_rows": 4488,
  "calibrated_book_score_rows": 408,
  "full_outcome_panel_loaded": false,
  "holdout_or_external_outcomes_loaded": false,
  "market_information_used": false,
  "probability_bridge_retained": false,
  "selected_parameters": [
    {
      "candidate_id": "catboost_quantile_pooled",
      "scale_factor": 2.0,
      "uniform_smoothing": 0.1
    },
    {
      "candidate_id": "gp_matern32_rule",
      "scale_factor": 1.5,
      "uniform_smoothing": 0.0
    },
    {
      "candidate_id": "pooled_empirical_residual",
      "scale_factor": 1.25,
      "uniform_smoothing": 0.0
    }
  ],
  "issue_rows": 0,
  "integrity_checks_passed": 8,
  "integrity_checks_to

,candidate_id,scale_factor,uniform_smoothing,date_balanced_mean_categorical_log_score,date_balanced_mean_multiclass_brier
0,catboost_quantile_pooled,2.00,0.1,1.625374,0.747230
1,gp_matern32_rule,1.50,0.0,1.542982,0.749762
2,pooled_empirical_residual,1.25,0.0,1.522599,0.740922


18wB development calibration release: PASS
